## Configuração da Página do Streamlit
📌 O que acontece aqui?

Define o título da página e o layout (wide = tela cheia).
Exibe o título principal "ARTEFACT" e uma descrição abaixo.

In [ ]:
st.set_page_config(page_title="ARTEFACT - Análise Preditiva", layout="wide")

st.title("ARTEFACT")
st.subheader("Análise preditiva utilizando modelo de previsão Prophet")

## Função para Download do CSV
📌 O que essa função faz?

Converte um DataFrame (df) em um arquivo CSV.
Codifica o CSV em Base64 para permitir o download no Streamlit.
Retorna um link HTML para baixar o arquivo com os resultados do forecast

In [ ]:
def download_csv(df):
    csv = df.to_csv(index=False)
    b64 = base64.b64encode(csv.encode()).decode()
    href = f'<a href="data:file/csv;base64,{b64}" download="forecast_results.csv">Download CSV File</a>'
    return href

### Função para Criar Gráfico de Forecast com Plotly
📌 O que essa função faz?

Cria um gráfico interativo usando plotly para visualizar a previsão gerada pelo Prophet.
Plota:
Histórico (azul)
Previsão (verde)
Limite superior (laranja, tracejado)
Limite inferior (vermelho, tracejado)
Retorna um gráfico que será exibido no Streamlit.

In [ ]:
def create_forecast_plot(forecast, col):
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=forecast[forecast['type'] == 'Histórico']["ds"], 
                             y=forecast[forecast['type'] == 'Histórico']["y"], 
                             mode='lines', name="Histórico", line=dict(color='blue', width=2)))
    fig.add_trace(go.Scatter(x=forecast[forecast['type'] == 'Forecast']["ds"], 
                             y=forecast[forecast['type'] == 'Forecast']["yhat"], 
                             mode='lines+markers', name="Previsão (yhat)", line=dict(color='green', width=2)))
    fig.add_trace(go.Scatter(x=forecast[forecast['type'] == 'Forecast']["ds"], 
                             y=forecast[forecast['type'] == 'Forecast']["yhat_upper"], 
                             mode='lines', name="Limite Superior", line=dict(color='orange', width=1, dash='dash')))
    fig.add_trace(go.Scatter(x=forecast[forecast['type'] == 'Forecast']["ds"], 
                             y=forecast[forecast['type'] == 'Forecast']["yhat_lower"], 
                             mode='lines', name="Limite Inferior", line=dict(color='red', width=1, dash='dash')))
    fig.update_layout(title=f"Forecast para {col}", 
                      xaxis_title="Data", 
                      yaxis_title="Valor", 
                      legend_title="Legenda", 
                      template="plotly_white")
    return fig

### Sidebar para Upload de Arquivo
📌 O que acontece aqui?

Adiciona um header "Configurações" na sidebar do Streamlit.
Permite que o usuário faça o upload de um arquivo (.csv, .xlsx ou .xls).

In [ ]:
st.sidebar.header("Configurações")
uploaded_file = st.sidebar.file_uploader("Arraste e solte a base de dados aqui", type=["csv", "xlsx", "xls"])

### Carregamento dos Dados
📌 O que acontece aqui?

Verifica se um arquivo foi carregado.
Lê os dados e os exibe na tela (10 primeiras linhas).
Se o formato não for suportado, exibe um erro e para a execução.

In [ ]:
if uploaded_file:
    try:
        if uploaded_file.name.endswith('.csv'):
            data = pd.read_csv(uploaded_file)
        elif uploaded_file.name.endswith(('.xlsx', '.xls')):
            data = pd.read_excel(uploaded_file)
        else:
            st.error("Formato de arquivo não suportado. Use CSV, XLS ou XLSX.")
            st.stop()

        st.subheader("Dados Carregados (10 Primeiras Linhas):")
        st.dataframe(data.head(10))

### Pré-processamento dos Dados
📌 O que acontece aqui?

Permite que o usuário trate valores ausentes (preenchendo com o valor anterior ou seguinte).
Oferece a opção de normalizar os dados usando MinMaxScaler (entre 0 e 1).

In [ ]:
st.sidebar.subheader("Pré-processamento")
handle_missing = st.sidebar.checkbox("Tratar valores ausentes")
if handle_missing:
    data = data.fillna(method='ffill').fillna(method='bfill')

normalize_data = st.sidebar.checkbox("Normalizar dados")
if normalize_data:
    scaler = MinMaxScaler()
    numeric_columns = data.select_dtypes(include=[np.number]).columns
    data[numeric_columns] = scaler.fit_transform(data[numeric_columns])

###  Configuração do Forecast
📌 O que acontece aqui?

O usuário seleciona a coluna de data.
Define o número de meses para previsão (1 a 24 meses).

In [ ]:
st.sidebar.subheader("Configuração do Forecast")
date_column = st.sidebar.selectbox("Selecione a coluna de data:", data.columns)
forecast_months = st.sidebar.slider("Meses para previsão:", 1, 24, 6)

### Configuração dos Parâmetros do Prophet
📌 O que acontece aqui?

Define se o modelo usará sazonalidade anual, semanal e diária.

In [ ]:
st.sidebar.subheader("Parâmetros do Prophet")
yearly_seasonality = st.sidebar.checkbox("Sazonalidade anual", value=True)
weekly_seasonality = st.sidebar.checkbox("Sazonalidade semanal", value=True)
daily_seasonality = st.sidebar.checkbox("Sazonalidade diária", value=False)

### Loop para Criar os Modelos Prophet
📌 O que acontece aqui?

Para cada coluna numérica, um modelo Prophet é criado e treinado.
O forecast é gerado e classificado como Histórico ou Forecast.
O gráfico interativo é exibido no Streamlit.

In [ ]:
for col in numeric_columns:
    with st.expander(f"Forecast para {col}"):
        forecast_data = data[[date_column, col]].rename(columns={date_column: "ds", col: "y"})

        model = Prophet(yearly_seasonality=yearly_seasonality,
                        weekly_seasonality=weekly_seasonality,
                        daily_seasonality=daily_seasonality)
        model.fit(forecast_data)

        future = model.make_future_dataframe(periods=forecast_months, freq="M")
        forecast = model.predict(future)

        last_date = forecast_data['ds'].max()
        forecast['y'] = forecast['ds'].map(dict(zip(forecast_data['ds'], forecast_data['y'])))
        forecast['type'] = forecast['ds'].apply(lambda x: 'Histórico' if x <= last_date else 'Forecast')
        forecast['y'] = forecast.apply(lambda row: row['y'] if row['type'] == 'Histórico' else row['yhat'], axis=1)
        forecast['item'] = col

        consolidated_forecast = pd.concat([consolidated_forecast, forecast], ignore_index=True)

        st.plotly_chart(create_forecast_plot(forecast, col), use_container_width=True)